In [1]:
# Setup: imports, env, data loading
import os
import re
from pathlib import Path
from dataclasses import dataclass, field
from typing import Callable, List, Dict, Any, Optional

# Disable HF Xet + SSL noise (consistent with other notebooks in this repo)
os.environ["HF_HUB_DISABLE_XET"] = "1"
import urllib3
urllib3.disable_warnings()

from dotenv import load_dotenv

env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)


True

In [15]:
# Identify text files in the Datasets folder and load one file
datasets_dir = Path(r"C:\projects\learn-rag\Datasets")
txt_files = sorted(datasets_dir.glob("*.txt"))

print(f"Found {len(txt_files)} text file(s) in {datasets_dir}:")
for i, file_path in enumerate(txt_files, start=1):
    print(f"{i}. {file_path.name}")

Found 4 text file(s) in C:\projects\learn-rag\Datasets:
1. amazon_2023.txt
2. amazon_2024.txt
3. microsoft_2023.txt
4. microsoft_2024.txt


In [16]:
if txt_files:
    selected_file = txt_files[0]
    text_data = selected_file.read_text(encoding="utf-8", errors="ignore")
    print(f"\nLoaded file: {selected_file.name}")
    print(f"Character count: {len(text_data):,}")
    print("Preview:\n")
    print(text_data[:500])
else:
    text_data = ""
    print("No .txt files found.")


Loaded file: amazon_2023.txt
Character count: 5,593
Preview:

company_name: Amazon
year: 2023
ticker: AMZN
fiscal_period: FY 2023
headquarters: Seattle, Washington, United States
ceo: Andy Jassy
founded: 1994
industry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI
revenue_usd_billions: 574.7
net_income_usd_billions: 30.4
employee_count: 1525000

Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was found


## 1. CharacterTextSplitter — beyond basic `chunk_size`

- Splits on a **single custom separator** (regex or literal).
- Useful when documents have a known, uniform delimiter (logs, CSV-like, fixed templates).
- Fastest option; falls apart on heterogeneous prose.


In [17]:
from langchain_text_splitters import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    separator="\n\n",
    chunk_size=250,
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=False,
)
texts = text_splitter.create_documents([text_data])
texts

Created a chunk of size 313, which is longer than the specified 250
Created a chunk of size 439, which is longer than the specified 250
Created a chunk of size 316, which is longer than the specified 250
Created a chunk of size 471, which is longer than the specified 250
Created a chunk of size 434, which is longer than the specified 250
Created a chunk of size 520, which is longer than the specified 250
Created a chunk of size 505, which is longer than the specified 250
Created a chunk of size 453, which is longer than the specified 250
Created a chunk of size 510, which is longer than the specified 250
Created a chunk of size 423, which is longer than the specified 250
Created a chunk of size 533, which is longer than the specified 250


[Document(metadata={}, page_content='company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000'),
 Document(metadata={}, page_content="Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS."),
 Document(metadata={}, page_content='In 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year reflec

In [37]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=250,
    chunk_overlap=50,
)

docs = splitter.create_documents(
    texts=[text_data],
    metadatas=[{
        "source": str(selected_file),
        "doc_name": selected_file.name,
        "year": "2023",
        "company": "amazon",
    }],
)

docs

[Document(metadata={'source': 'C:\\projects\\learn-rag\\Datasets\\amazon_2023.txt', 'doc_name': 'amazon_2023.txt', 'year': '2023', 'company': 'amazon'}, page_content="company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000\n\nAmazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.\n\nIn 2023, Amazon reported approximately $574.7 billion in revenu

## 2. RecursiveCharacterTextSplitter — natural-boundary fallback

- Tries an ordered list of separators: paragraphs → lines → sentences → words → chars.
- Stops at the first level that keeps chunks ≤ `chunk_size`; falls back if too big.
- Best general-purpose default for plain prose.


In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=50)
texts = text_splitter.split_text(text_data)
texts

['company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI',
 'revenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000',
 'Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief',
 "Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.",
 'In 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year reflected a strong recovery in profitability after the weake

In [20]:
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=450,
    chunk_overlap=60,
)
recursive_chunks = recursive_splitter.split_text(text_data)
recursive_chunks

['company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000',
 "Amazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services. The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023. Amazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.",
 'In 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income. The year reflected a strong recovery in profitability after the weaker 2022 period. Amazon continued to focus on operation

Spacy sentence splitter

In [21]:
from langchain_text_splitters import SpacyTextSplitter

spacy_splitter = SpacyTextSplitter(
    pipeline="sentencizer",
    chunk_size=500,
    chunk_overlap=75,
)
spacy_chunks = spacy_splitter.split_text(text_data)
spacy_chunks

Created a chunk of size 771, which is longer than the specified 500


['company_name: Amazon\nyear: 2023\nticker: AMZN\nfiscal_period: FY 2023\nheadquarters: Seattle, Washington, United States\nceo: Andy Jassy\nfounded: 1994\nindustry: E-commerce, cloud computing, digital advertising, logistics, streaming, AI\nrevenue_usd_billions: 574.7\nnet_income_usd_billions: 30.4\nemployee_count: 1525000\n\nAmazon is a multinational technology company best known for e-commerce, Amazon Web Services (AWS), digital advertising, logistics, devices, and streaming services.',
 "The company was founded by Jeff Bezos in 1994, and Andy Jassy served as chief executive officer in 2023.\n\nAmazon's operating model combines first-party retail, a large third-party marketplace, subscription services such as Prime, and a fast-growing cloud business through AWS.\n\n\n\nIn 2023, Amazon reported approximately $574.7 billion in revenue and about $30.4 billion in net income.\n\nThe year reflected a strong recovery in profitability after the weaker 2022 period.",
 "Amazon continued to fo